# Does simulated disagreement predict real controversy?

A backtest for [Lightningfish](https://github.com/rajul-kk/LightningFish) that
scores something every previous run threw away — and that already broke once,
which is why this notebook calibrates instead of guessing.

## Why this exists

Every earlier backtest reduced a finished simulation to `sign(final mean opinion)`
— **one bit**, and the same bit a single LLM call produces. Scored that way, at
n=200, the result was unambiguous:

| | accuracy |
|---|---|
| author-karma heuristic | 62.5% |
| **simulation** | **51.5%** |
| single LLM call | 50.0% |
| majority class | 50.0% |

`p = 0.9994`. The simulation is at chance. It beats one raw model call by 1.5
points — a real but negligible contribution from all the agents and rounds.

Here is the problem with concluding "the engine is worthless" from that: we
evaluated a *population* on the one output where a population has no structural
advantage. A multi-agent run also produces a **distribution** — 24 opinions with
a spread — and nothing about the mean captures whether the crowd agreed.

This notebook tests the other moment: does the dispersion of simulated opinion
predict whether a real HN thread turned into an argument?

## What went wrong the first time, and why this version is different

The first attempt (n=107) reported 53.3% accuracy and looked unremarkable —
until two problems surfaced:

1. **A real bug.** `CachingAdapter` (used by the CLI for every HN run) didn't
   delegate the new dispersion-scoring hook, so it silently fell back to
   scoring the mean axis again. The 53.3% never measured controversy at all.
   Fixed in the engine — `CachingAdapter` now delegates every `DomainAdapter`
   method, enforced by a structural test.
2. **A guessed threshold.** The dispersion cutoff (`stddev >= 0.35`) was a bare
   guess with no basis. The real stddev range on that run was 0.085–0.286 —
   **never once reaching 0.35** — so even scored correctly, every event would
   have predicted the same class. A threshold that never fires isn't a test.

**This notebook fixes both.** It calibrates the threshold from a held-out
*calibration* batch (median of their simulated stddevs) and reports accuracy
only on a disjoint *evaluation* batch — so the threshold is chosen before, and
independently of, the data it's scored against
(see [METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md)
rule 3 and rule 6).

## Honest framing before you run it

- This is **not** a claim a single call cannot make in principle — you can just
  ask a model "will this be controversial" (the `single_llm` rung does exactly
  that). The simulation *derives* disagreement from population heterogeneity
  rather than asserting it; that's a real but narrower distinction than
  "structurally impossible."
- Controversy prediction is an established task elsewhere (Reddit, Wikipedia
  edit wars). Doing it on HN isn't novel by itself.
- Given the mean axis is settled at chance, **expect this to fail too.** The
  point is closing the question of whether the standard evaluation was simply
  measuring the wrong output, not rescuing a result.

---
## What the data is

**Source:** the [Hacker News Algolia API](https://hn.algolia.com/api) — free,
unauthenticated, ~10k requests/hour. No key, no scraping.

**Sample:** settled stories at least 24 hours old, pulled class-balanced (half
above the high points threshold, half below the low one).

**The seed each agent reads** — strictly submission-time fields, never the
outcome:

| Field | Example |
|---|---|
| title | "China is now the world's greatest oil power" |
| author + karma | `bookofjoe` (110,566) |
| url domain | `economist.com` |
| type | story / Ask HN / Show HN |
| self-text | first 500 chars, if any |

**The label:** `num_comments / points` at settlement.

- ratio >= 0.7 -> **contested** (a thread arguing with itself)
- ratio < 0.4 -> **consensus** (quietly upvoted)
- in between -> skipped, no clean signal
- **fewer than 20 points -> skipped entirely**, because 0 comments on a 1-point
  story means nobody saw it, not that everyone agreed.

This floor and the mid-ratio gap remove most pulled stories — expect roughly
1 in 3 to be usable, which is why `PULL_LIMIT` below is large.

Point-in-time safety is enforced in code: the seed enricher is a separate
module from the ground-truth fetcher, and tests assert the target values never
appear in seed text or metadata.

---
## What the model is

**Inference:** [qwen2.5:7b](https://ollama.com/library/qwen2.5) (Q4_K_M, ~4.7 GB)
served locally by Ollama inside the notebook. No API keys, $0. It fits a T4's
16 GB of VRAM — the reason this runs here: on a loaded 16 GB CPU box the same
job completed **zero** events in 80 minutes.

**The simulation** runs a population of 24 agents over 3–4 rounds, split into
three tiers each round:

| Tier | Share | What happens |
|---|---|---|
| T1 originators | ~10% | LLM writes a structured post |
| T2 reactors | ~20% | LLM re-evaluates after reading a feed of others' posts |
| T3 drifters | rest | deterministic herding maths, no LLM call |

**The agents** are six HN archetypes with different resistance, recency bias,
contrarian tendency and herding coefficients. Heterogeneity is the point — a
population of identical agents would converge trivially and its spread would
carry no information.

**What is scored:** the standard deviation of the 24 final opinions, against a
threshold **calibrated by this run**, not hardcoded (see below).

**The baseline ladder** — the simulation must beat every rung:

| Rung | What it controls for |
|---|---|
| majority class | degenerate data |
| `naive` | Ask-HN / question-mark heuristic, no model |
| `single_llm` | one call asked *the same controversy question* |
| the simulation | — |

---
## 1. Setup

Sidebar: **Accelerator -> GPU** and **Internet -> On**.

`zstd` first: Ollama's installer needs it to extract, Kaggle's image doesn't
ship it, and without this the install fails quietly and surfaces later as a
confusing `FileNotFoundError: 'ollama'`.

In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")

In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")

In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only; also confirms the CachingAdapter delegation fix
# and the calibrated-threshold code are actually present in this clone.
!python -m pytest tests/core tests/hn -q 2>&1 | tail -3

---
## 2. Configuration

`PULL_LIMIT` needs to be generous: the 20-point floor plus the mid-ratio gap
zone discard roughly 2 of every 3 pulled stories, and the calibration/
evaluation split then halves what's left again. Aim for at least ~40 usable
events per half.

In [ ]:
PULL_LIMIT = 300      # stories to pull; expect ~1 in 3 to be scoreable
N_AGENTS   = 24        # population size - the spread of THIS is what's scored
N_ROUNDS   = 4

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | pulling {PULL_LIMIT}")

### Throughput check

~26 model calls per event. Confirm the per-call cost before committing — if
this shows double digits you're on CPU regardless of what the assert said.

In [ ]:
import time
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
print(f"{per_call:.2f}s per call  ->  ~{per_call*26:.0f}s per event")

!python -m tests.integration.run_backtest hn-controversy-calibrated {PULL_LIMIT} 2>&1 | tee /kaggle/working/hn_controversy_calibrated.log


In [ ]:
!python -m tests.integration.run_backtest hn-controversy-calibrated {PULL_LIMIT} 2>&1 | tee /kaggle/working/hn_controversy_calibrated.log


### Reading the log

- `X of Y events have a controversy direction` — how much the points floor and
  gap zone discarded.
- `split: N calibration / M evaluation` — abort if either is under ~15; not
  enough basis to calibrate or evaluate.
- `calibration stddev range: a-b, median (threshold) = t` — the derived cutoff.
  Compare it to the 0.35 that failed last time.
- The final report block: `beats_baselines` must PASS on every rung, and
  `p_value_vs_best` below 0.05, for this to be a result rather than noise.
- **A WARNING about a constant predictor on the eval set** would mean the two
  halves have different stddev distributions, or n is still too small — treat
  that as inconclusive, not a negative.

---
## 4. Save

In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/cache

The cache holds every simulated run's final distribution, so a future question
about these same events — a different threshold, a different axis — costs
nothing to re-score.

## Result interpretation

A negative here is the expected outcome and still worth recording: the mean
axis is settled at chance across n=200, and if a *properly calibrated*
dispersion axis also carries nothing, that closes the question of whether the
standard evaluation was measuring the wrong output — cleanly, this time,
without a bug or a guessed threshold muddying the answer.

Either way the number belongs in
[METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md),
alongside the rest.